# G1: repository delivery metrics (`01_repository_delivery_metrics`)

**Question.** For each repository and day: how many pull request events happened, what share of the closed ones were merged, how long did pull requests take from creation to merge and to close, and how big were they?

This notebook builds the Gold table `github_lakehouse.gold.repository_delivery_metrics`.

Contract: `docs/gold_contracts.md`, section 9. The way a Gold table is refreshed is in section 6.

| | |
|---|---|
| Reads | `silver.pull_request_events` only |
| Writes | `gold.repository_delivery_metrics`, its staging copy `gold_staging.repository_delivery_metrics`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per repository per UTC day on which it had at least one pull request event |
| Key | `repo_id`, `activity_date` |

The notebook has the same eleven steps as `02_repository_issue_flow`. Steps 1, 3, 6, 8 and 10 are the same code with other table names. What is new is in steps 2, 5 and 7.

## Two families of numbers

- **Event numbers** count events. A pull request that is closed, reopened and closed again on one day has two close actions.
- **Pull request numbers** count each pull request once per day. Its latest event of that day is its **snapshot**. The size averages use snapshots, so a pull request with four events is not counted four times.

## The rules of this table

- **Merge share** uses rows with the action `closed` and a known `pr_merged`.
- **Merge hours** use rows with the action `closed`, `pr_merged` true, both times known, and a merge time that is not earlier than the creation time.
- **Close hours** use rows with the action `closed`, both times known, and a close time that is not earlier than the creation time.
- **A duration belongs to the day of the close event**, even when the pull request was created months earlier.
- **A pull request is identified** by `pr_id`. If that is missing, by `pr_number` in its repository. If both are missing, the event still counts as an event, but not as a pull request.
- **Each size average** uses only the snapshots where that size is known.
- **NULL is not zero.** A share, a duration or an average without eligible rows is NULL, and the eligible count next to it is 0.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Affected days | Decides which days this run rebuilds |
| 4 | Read Silver | Only the columns the table needs, only the affected days |
| 5 | Build | Hours and pull request key per event, event numbers, snapshots, size numbers, the name label |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy |
| 8 | Publish | Replaces the affected days in the Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | All days are rebuilt and the whole Gold table is replaced | `YYYY-MM-DD`: only the days touched by that source date are rebuilt and replaced |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_repository_delivery_metrics"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The four table names are set **only** in this cell. Later cells never change them, otherwise a test run could write to production.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_TABLE = "github_lakehouse.silver.pull_request_events"
    GOLD_TABLE = "github_lakehouse.gold.repository_delivery_metrics"
    STAGING_TABLE = "github_lakehouse.gold_staging.repository_delivery_metrics"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_TABLE = f"github_lakehouse.{test_schema}.pull_request_events"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.repository_delivery_metrics"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.repository_delivery_metrics_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

print("Silver table :", SILVER_TABLE)
print("Gold table   :", GOLD_TABLE)
print("Staging table:", STAGING_TABLE)
print("Audit table  :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`, and the Silver table by `Silver_layer/02_pull_request_events`. If one of them does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

if not spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        f"{SILVER_TABLE} does not exist. Run the Silver notebook 02_pull_request_events first."
    )

print("Setup found:", AUDIT_TABLE)
print("Source found:", SILVER_TABLE)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `activity_date` | Key. The UTC date of the events |
| `repo_name` | Label: the latest name seen that day |
| `pr_event_count` | All pull request events that day |
| `pr_action_known_rows` | Events with a known action |
| `pr_opened_actions` | Events with the action `opened` |
| `pr_closed_actions` | Events with the action `closed` |
| `pr_reopened_actions` | Events with the action `reopened` |
| `pr_other_action_rows` | Events with a known action that is none of the three. Expected 0 |
| `eligible_closed_rows` | Closed events with a known merged flag. The base of the merge share |
| `merged_closed_rows` | Of those, the merged ones |
| `merge_share_pct` | 100 × merged ÷ eligible. NULL when eligible is 0 |
| `merged_duration_eligible_rows` | Merged close events with usable times. The base of the two columns below |
| `avg_merge_hours` | Average hours from creation to merge |
| `p50_merge_hours` | Median hours from creation to merge |
| `closed_duration_eligible_rows` | Close events with usable times. The base of the two columns below |
| `avg_close_hours` | Average hours from creation to close |
| `p50_close_hours` | Median hours from creation to close |
| `pr_size_snapshot_rows` | Different pull requests seen that day, one snapshot each |
| `pr_commits_eligible_rows` | Snapshots with a known commit count |
| `avg_pr_commits` | Average commits over those snapshots |
| `pr_additions_eligible_rows` | Snapshots with a known number of added lines |
| `avg_pr_additions` | Average added lines |
| `pr_deletions_eligible_rows` | Snapshots with a known number of deleted lines |
| `avg_pr_deletions` | Average deleted lines |
| `pr_changed_files_eligible_rows` | Snapshots with a known number of changed files |
| `avg_pr_changed_files` | Average changed files |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

Every share, duration and average has a count next to it that says how many rows it is built from.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and label
        repo_id                         BIGINT,
        activity_date                   DATE,
        repo_name                       STRING,

        -- Events by action
        pr_event_count                  BIGINT,
        pr_action_known_rows            BIGINT,
        pr_opened_actions               BIGINT,
        pr_closed_actions               BIGINT,
        pr_reopened_actions             BIGINT,
        pr_other_action_rows            BIGINT,

        -- Merge share
        eligible_closed_rows            BIGINT,
        merged_closed_rows              BIGINT,
        merge_share_pct                 DOUBLE,

        -- Time from creation to merge
        merged_duration_eligible_rows   BIGINT,
        avg_merge_hours                 DOUBLE,
        p50_merge_hours                 DOUBLE,

        -- Time from creation to close
        closed_duration_eligible_rows   BIGINT,
        avg_close_hours                 DOUBLE,
        p50_close_hours                 DOUBLE,

        -- Size, one snapshot per pull request and day
        pr_size_snapshot_rows           BIGINT,
        pr_commits_eligible_rows        BIGINT,
        avg_pr_commits                  DOUBLE,
        pr_additions_eligible_rows      BIGINT,
        avg_pr_additions                DOUBLE,
        pr_deletions_eligible_rows      BIGINT,
        avg_pr_deletions                DOUBLE,
        pr_changed_files_eligible_rows  BIGINT,
        avg_pr_changed_files            DOUBLE,

        -- Technical columns
        _gold_run_id                    STRING,
        _gold_updated_at                TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Affected days

A Gold row is a summary of a whole day. When one new event arrives for a day, the row of that day has to be calculated again from **all** events of that day. So the first question of every run is: which days does this run have to rebuild?

### 3.1 The Silver rows in the scope of this run

- With a `source_date`, the scope is the Silver rows that were loaded from that source date.
- With a blank date, the scope is all Silver rows.

A row without an event date cannot be placed on a day. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_row = spark.sql(f"""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM {SILVER_TABLE}
    WHERE {scope_filter}
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter                :", scope_filter)
print("Silver rows in the scope    :", rows_in_scope)
print("Rows without an event date  :", rows_without_event_date)

### 3.2 The days these rows belong to

The affected days are the event dates of the rows in the scope. In the week that is loaded, the event date and the source date of a row are always the same day (measured by the profiling run). So a run for one source date rebuilds exactly one day.

In [0]:
day_rows = spark.sql(f"""
    SELECT DISTINCT event_date
    FROM {SILVER_TABLE}
    WHERE {scope_filter}
      AND event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

print("Affected days:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)

### 3.3 The day list as SQL text

The later steps need the affected days inside a SQL `IN (...)` list, for example `DATE'2024-02-01', DATE'2024-02-02'`. The same text is used to read Silver, to check the result and to publish, so all three work on exactly the same days.

When there is no affected day, the text is `NULL`. `IN (NULL)` is never true, so the later queries simply find no row.

In [0]:
if len(affected_dates) == 0:
    affected_days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    affected_days_sql = ", ".join(day_texts)

print("Day list:", affected_days_sql)

## 4. Read Silver

All Silver rows of the affected days are read, **not only the rows of this source date**. A row that arrived earlier for the same day must be counted again, otherwise the new Gold row would be too small.

Only the columns this table needs are selected. `event_date` gets its Gold name `activity_date`.

Spark does not read anything yet. A temporary view is only a name for a query. The work is done in step 6, when the result is written.

In [0]:
pr_rows = spark.sql(f"""
    SELECT
        event_id,
        event_timestamp,
        event_date  AS activity_date,
        repo_id,
        repo_name,
        pr_action,
        pr_id,
        pr_number,
        pr_created_at,
        pr_closed_at,
        pr_merged_at,
        pr_merged,
        pr_commits,
        pr_additions,
        pr_deletions,
        pr_changed_files
    FROM {SILVER_TABLE}
    WHERE event_date IN ({affected_days_sql})
""")
pr_rows.createOrReplaceTempView("pr_rows")

### 4.1 How many Silver rows these days have

The rows are counted directly on the Silver table, not on the view above. So this number does not depend on the steps that follow.

It goes into the audit row. The quality gates compare it with the sum of `pr_event_count`: every Silver row of the affected days must be counted exactly once.

In [0]:
silver_row_count = spark.sql(f"""
    SELECT COUNT(*) AS silver_rows
    FROM {SILVER_TABLE}
    WHERE event_date IN ({affected_days_sql})
""").collect()[0]["silver_rows"]

print("Silver rows on the affected days:", silver_row_count)

## 5. Build

The Gold rows are built in six small queries. Each one is saved as a temporary view, and a later one reads it.

| Query | View | One row per |
|---|---|---|
| 5.1 | `pr_events_prepared` | Event, with its hours and its pull request key |
| 5.2 | `pr_event_counts` | Repository and day: the event numbers |
| 5.3 | `pr_snapshots` | Pull request and day: its latest event |
| 5.4 | `pr_size_counts` | Repository and day: the size numbers |
| 5.5 | `repo_names` | Repository and day: the name label |
| 5.6 | `gold_rows` | Repository and day: everything together |

### 5.1 Hours and pull request key of each event

Three values are worked out for every event. The row count does not change in this step.

**`merge_hours`** is filled only for an eligible merged close event:

1. the action is `closed`,
2. `pr_merged` is true,
3. the creation time and the merge time are both known,
4. the merge time is not earlier than the creation time.

**`close_hours`** is filled only for an eligible close event:

1. the action is `closed`,
2. the creation time and the close time are both known,
3. the close time is not earlier than the creation time.

For every other event they stay NULL. A `CASE` without `ELSE` gives NULL.

The hours are the seconds between the two times, divided by 3,600. `unix_timestamp` turns a time into a number of seconds. The time of the event itself is not used.

Why the action must be `closed`: an event carries the state of the pull request at a slightly later moment. In this week 4,904 events that are not `closed` already say `pr_merged = true`, and 5,542 `opened` events already carry a close time. Without this rule one merged pull request could be counted twice.

**`pr_key`** identifies the pull request: the `pr_id`, or the `pr_number` when the ID is missing. The number is unique only inside one repository. That is enough here, because the later steps always work per repository. If both are missing, the key stays NULL.

In [0]:
pr_events_prepared = spark.sql("""
    SELECT
        event_id,
        event_timestamp,
        repo_id,
        activity_date,
        pr_action,
        pr_merged,
        pr_commits,
        pr_additions,
        pr_deletions,
        pr_changed_files,

        CASE
            WHEN pr_action = 'closed'
             AND pr_merged = true
             AND pr_created_at IS NOT NULL
             AND pr_merged_at IS NOT NULL
             AND pr_merged_at >= pr_created_at
            THEN (unix_timestamp(pr_merged_at) - unix_timestamp(pr_created_at)) / 3600
        END AS merge_hours,

        CASE
            WHEN pr_action = 'closed'
             AND pr_created_at IS NOT NULL
             AND pr_closed_at IS NOT NULL
             AND pr_closed_at >= pr_created_at
            THEN (unix_timestamp(pr_closed_at) - unix_timestamp(pr_created_at)) / 3600
        END AS close_hours,

        CASE
            WHEN pr_id IS NOT NULL
            THEN concat('id:', CAST(pr_id AS STRING))
            WHEN pr_number IS NOT NULL
            THEN concat('number:', CAST(pr_number AS STRING))
        END AS pr_key

    FROM pr_rows
""")
pr_events_prepared.createOrReplaceTempView("pr_events_prepared")

### 5.2 Event numbers per repository and day

One `GROUP BY` gives one row per repository and day, with all the numbers that count events.

How the functions treat NULL:

- `COUNT(*)` counts every row.
- `count_if(condition)` counts the rows where the condition is true. A row with a NULL action, or a NULL `pr_merged`, is true for none of the conditions.
- `COUNT(column)` counts only the rows where the column is not NULL. So `COUNT(merge_hours)` is the number of eligible merged close events.
- `AVG` and `median` use only the rows where the value is not NULL. With no such row they give NULL, not 0.

`median` is the exact median. With an even number of values it is the average of the two middle values: the median of 1, 2, 3 and 10 hours is 2.5 hours.

`NOT IN ('opened', 'closed', 'reopened')` finds a known action that is none of the three. None is expected.

The averages and the medians are not rounded here. They are rounded once, in step 5.6.

In [0]:
pr_event_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,

        COUNT(*)                                 AS pr_event_count,
        count_if(pr_action IS NOT NULL)          AS pr_action_known_rows,
        count_if(pr_action = 'opened')           AS pr_opened_actions,
        count_if(pr_action = 'closed')           AS pr_closed_actions,
        count_if(pr_action = 'reopened')         AS pr_reopened_actions,
        count_if(pr_action NOT IN ('opened', 'closed', 'reopened'))
                                                 AS pr_other_action_rows,

        count_if(pr_action = 'closed' AND pr_merged IS NOT NULL)
                                                 AS eligible_closed_rows,
        count_if(pr_action = 'closed' AND pr_merged = true)
                                                 AS merged_closed_rows,

        COUNT(merge_hours)                       AS merged_duration_eligible_rows,
        AVG(merge_hours)                         AS avg_merge_hours_exact,
        median(merge_hours)                      AS p50_merge_hours_exact,

        COUNT(close_hours)                       AS closed_duration_eligible_rows,
        AVG(close_hours)                         AS avg_close_hours_exact,
        median(close_hours)                      AS p50_close_hours_exact

    FROM pr_events_prepared
    GROUP BY repo_id, activity_date
""")
pr_event_counts.createOrReplaceTempView("pr_event_counts")

### 5.3 One snapshot per pull request and day

Every event of a pull request carries the size of that pull request. If all events were averaged, a pull request with four events would count four times. So each pull request is taken once per day: its **latest** event of that day.

`ROW_NUMBER` numbers the events of each pull request on each day, the latest first. Number 1 is the snapshot.

- If two events have the same time, the one with the larger `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.
- Events without a pull request key are left out. They are events, but not pull requests.

In [0]:
pr_snapshots = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        pr_key,
        pr_commits,
        pr_additions,
        pr_deletions,
        pr_changed_files
    FROM (
        SELECT
            repo_id,
            activity_date,
            pr_key,
            pr_commits,
            pr_additions,
            pr_deletions,
            pr_changed_files,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date, pr_key
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS event_rank
        FROM pr_events_prepared
        WHERE pr_key IS NOT NULL
    ) ranked_events
    WHERE event_rank = 1
""")
pr_snapshots.createOrReplaceTempView("pr_snapshots")

### 5.4 Size numbers per repository and day

The snapshots are counted and averaged per repository and day.

Each size has its own count. `COUNT(pr_commits)` counts only the snapshots where the commit count is known, and `AVG(pr_commits)` uses the same snapshots. A snapshot with a known number of added lines but an unknown commit count is used for the one average and left out of the other.

A size of 0 is a value. Only NULL is left out.

In [0]:
pr_size_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,

        COUNT(*)                 AS pr_size_snapshot_rows,

        COUNT(pr_commits)        AS pr_commits_eligible_rows,
        AVG(pr_commits)          AS avg_pr_commits_exact,

        COUNT(pr_additions)      AS pr_additions_eligible_rows,
        AVG(pr_additions)        AS avg_pr_additions_exact,

        COUNT(pr_deletions)      AS pr_deletions_eligible_rows,
        AVG(pr_deletions)        AS avg_pr_deletions_exact,

        COUNT(pr_changed_files)  AS pr_changed_files_eligible_rows,
        AVG(pr_changed_files)    AS avg_pr_changed_files_exact

    FROM pr_snapshots
    GROUP BY repo_id, activity_date
""")
pr_size_counts.createOrReplaceTempView("pr_size_counts")

### 5.5 The `repo_name` label

A repository can be renamed, so `repo_name` is only a label and never a key. The label of a day is the name on the **latest** event of that repository on that day.

`ROW_NUMBER` numbers the events of each repository and day, the latest first. Number 1 is the one we keep.

- Events without a name are left out first, so a missing name is skipped.
- If two events have the same time, the one with the larger `event_id` gets number 1.

In [0]:
repo_names = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        repo_name
    FROM (
        SELECT
            repo_id,
            activity_date,
            repo_name,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS name_rank
        FROM pr_rows
        WHERE repo_name IS NOT NULL
    ) ranked_names
    WHERE name_rank = 1
""")
repo_names.createOrReplaceTempView("repo_names")

### 5.6 The Gold rows

The event numbers, the size numbers and the label are put together. The columns are in the order of the Gold table.

- The event numbers are the base: they have one row for every repository and day with an event.
- `LEFT JOIN` keeps a repository-day that has no snapshot, or no name. Without a snapshot the five size counts would be NULL, so `COALESCE` turns them into 0. The size averages stay NULL.
- **Merge share:** 100 × merged ÷ eligible. In SQL, `/` gives a decimal number also when both sides are whole numbers. The `CASE` leaves the share NULL when there is no eligible row. It also keeps the query from dividing by zero.
- The share, the averages and the medians are rounded to 2 decimals here, once.
- The two technical columns say which run wrote the row and when.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        c.repo_id,
        c.activity_date,
        n.repo_name,

        c.pr_event_count,
        c.pr_action_known_rows,
        c.pr_opened_actions,
        c.pr_closed_actions,
        c.pr_reopened_actions,
        c.pr_other_action_rows,

        c.eligible_closed_rows,
        c.merged_closed_rows,
        CASE
            WHEN c.eligible_closed_rows > 0
            THEN ROUND(100 * c.merged_closed_rows / c.eligible_closed_rows, 2)
        END AS merge_share_pct,

        c.merged_duration_eligible_rows,
        ROUND(c.avg_merge_hours_exact, 2)  AS avg_merge_hours,
        ROUND(c.p50_merge_hours_exact, 2)  AS p50_merge_hours,

        c.closed_duration_eligible_rows,
        ROUND(c.avg_close_hours_exact, 2)  AS avg_close_hours,
        ROUND(c.p50_close_hours_exact, 2)  AS p50_close_hours,

        COALESCE(s.pr_size_snapshot_rows, 0)           AS pr_size_snapshot_rows,
        COALESCE(s.pr_commits_eligible_rows, 0)        AS pr_commits_eligible_rows,
        ROUND(s.avg_pr_commits_exact, 2)               AS avg_pr_commits,
        COALESCE(s.pr_additions_eligible_rows, 0)      AS pr_additions_eligible_rows,
        ROUND(s.avg_pr_additions_exact, 2)             AS avg_pr_additions,
        COALESCE(s.pr_deletions_eligible_rows, 0)      AS pr_deletions_eligible_rows,
        ROUND(s.avg_pr_deletions_exact, 2)             AS avg_pr_deletions,
        COALESCE(s.pr_changed_files_eligible_rows, 0)  AS pr_changed_files_eligible_rows,
        ROUND(s.avg_pr_changed_files_exact, 2)         AS avg_pr_changed_files,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at

    FROM pr_event_counts c
    LEFT JOIN pr_size_counts s
      ON  s.repo_id = c.repo_id
      AND s.activity_date = c.activity_date
    LEFT JOIN repo_names n
      ON  n.repo_id = c.repo_id
      AND n.activity_date = c.activity_date
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM gold_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 7. Quality gates

The checks run on the staging copy. If one of them fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and day appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `activity_date` |
| `COUNT_IS_NULL` | One of the 15 count columns is missing. A count is 0 when nothing was counted, never NULL |
| `ACTIONS_DO_NOT_ADD_UP` | Opened + closed + reopened + other is not the number of known actions |
| `KNOWN_ACTIONS_ABOVE_EVENTS` | A row has more known actions than events |
| `CLOSED_COUNTS_OUT_OF_ORDER` | Merged ≤ eligible closed ≤ closed actions is not true |
| `SHARE_OUT_OF_RANGE` | A merge share is below 0 or above 100 |
| `DURATION_ROWS_ABOVE_THEIR_BASE` | More merge durations than merged closes, or more close durations than close actions |
| `NEGATIVE_DURATION` | An average or a median of hours is below 0 |
| `SNAPSHOT_COUNTS_OUT_OF_ORDER` | More snapshots than events, or a size count above the number of snapshots |
| `VALUE_WITHOUT_ELIGIBLE_ROWS` | A share, a duration or a size average is filled although its count is 0 |
| `ELIGIBLE_ROWS_WITHOUT_VALUE` | A count is above 0 but its share, duration or size average is NULL |
| `EVENT_SUM_DIFFERS_FROM_SILVER` | The sum of `pr_event_count` is not the number of Silver rows on the affected days |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, activity_date
            FROM {STAGING_TABLE}
            GROUP BY repo_id, activity_date
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last two values are the row count and the sum of all events.

The last two rules compare every value with its count, in both directions. Seven values have a count: the merge share, the merge hours, the close hours and the four size averages.

`COALESCE(SUM(...), 0)`: the sum of no rows is NULL. For the comparison with Silver it has to be 0.

In [0]:
failed_checks = []
staged_row_count = 0
staged_event_sum = 0

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR activity_date IS NULL)
                AS rows_without_key,

            count_if(pr_event_count IS NULL
                  OR pr_action_known_rows IS NULL
                  OR pr_opened_actions IS NULL
                  OR pr_closed_actions IS NULL
                  OR pr_reopened_actions IS NULL
                  OR pr_other_action_rows IS NULL
                  OR eligible_closed_rows IS NULL
                  OR merged_closed_rows IS NULL
                  OR merged_duration_eligible_rows IS NULL
                  OR closed_duration_eligible_rows IS NULL
                  OR pr_size_snapshot_rows IS NULL
                  OR pr_commits_eligible_rows IS NULL
                  OR pr_additions_eligible_rows IS NULL
                  OR pr_deletions_eligible_rows IS NULL
                  OR pr_changed_files_eligible_rows IS NULL)
                AS rows_with_missing_count,

            count_if(pr_opened_actions + pr_closed_actions
                     + pr_reopened_actions + pr_other_action_rows
                     <> pr_action_known_rows)
                AS rows_where_actions_do_not_add_up,

            count_if(pr_action_known_rows > pr_event_count)
                AS rows_with_known_actions_above_events,

            count_if(merged_closed_rows > eligible_closed_rows
                  OR eligible_closed_rows > pr_closed_actions)
                AS rows_with_closed_counts_out_of_order,

            count_if(merge_share_pct < 0 OR merge_share_pct > 100)
                AS rows_with_share_out_of_range,

            count_if(merged_duration_eligible_rows > merged_closed_rows
                  OR closed_duration_eligible_rows > pr_closed_actions)
                AS rows_with_duration_rows_above_base,

            count_if(avg_merge_hours < 0 OR p50_merge_hours < 0
                  OR avg_close_hours < 0 OR p50_close_hours < 0)
                AS rows_with_negative_duration,

            count_if(pr_size_snapshot_rows > pr_event_count
                  OR pr_commits_eligible_rows > pr_size_snapshot_rows
                  OR pr_additions_eligible_rows > pr_size_snapshot_rows
                  OR pr_deletions_eligible_rows > pr_size_snapshot_rows
                  OR pr_changed_files_eligible_rows > pr_size_snapshot_rows)
                AS rows_with_snapshot_counts_out_of_order,

            count_if(
                (eligible_closed_rows = 0 AND merge_share_pct IS NOT NULL)
                OR (merged_duration_eligible_rows = 0
                    AND (avg_merge_hours IS NOT NULL OR p50_merge_hours IS NOT NULL))
                OR (closed_duration_eligible_rows = 0
                    AND (avg_close_hours IS NOT NULL OR p50_close_hours IS NOT NULL))
                OR (pr_commits_eligible_rows = 0 AND avg_pr_commits IS NOT NULL)
                OR (pr_additions_eligible_rows = 0 AND avg_pr_additions IS NOT NULL)
                OR (pr_deletions_eligible_rows = 0 AND avg_pr_deletions IS NOT NULL)
                OR (pr_changed_files_eligible_rows = 0 AND avg_pr_changed_files IS NOT NULL)
            )   AS rows_with_value_without_eligible_rows,

            count_if(
                (eligible_closed_rows > 0 AND merge_share_pct IS NULL)
                OR (merged_duration_eligible_rows > 0
                    AND (avg_merge_hours IS NULL OR p50_merge_hours IS NULL))
                OR (closed_duration_eligible_rows > 0
                    AND (avg_close_hours IS NULL OR p50_close_hours IS NULL))
                OR (pr_commits_eligible_rows > 0 AND avg_pr_commits IS NULL)
                OR (pr_additions_eligible_rows > 0 AND avg_pr_additions IS NULL)
                OR (pr_deletions_eligible_rows > 0 AND avg_pr_deletions IS NULL)
                OR (pr_changed_files_eligible_rows > 0 AND avg_pr_changed_files IS NULL)
            )   AS rows_with_eligible_rows_without_value,

            COUNT(*)                          AS staged_rows,
            COALESCE(SUM(pr_event_count), 0)  AS staged_event_sum

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]
    staged_event_sum = gate_row["staged_event_sum"]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_actions_do_not_add_up"] > 0:
        failed_checks.append("ACTIONS_DO_NOT_ADD_UP")
    if gate_row["rows_with_known_actions_above_events"] > 0:
        failed_checks.append("KNOWN_ACTIONS_ABOVE_EVENTS")
    if gate_row["rows_with_closed_counts_out_of_order"] > 0:
        failed_checks.append("CLOSED_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_with_share_out_of_range"] > 0:
        failed_checks.append("SHARE_OUT_OF_RANGE")
    if gate_row["rows_with_duration_rows_above_base"] > 0:
        failed_checks.append("DURATION_ROWS_ABOVE_THEIR_BASE")
    if gate_row["rows_with_negative_duration"] > 0:
        failed_checks.append("NEGATIVE_DURATION")
    if gate_row["rows_with_snapshot_counts_out_of_order"] > 0:
        failed_checks.append("SNAPSHOT_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_with_value_without_eligible_rows"] > 0:
        failed_checks.append("VALUE_WITHOUT_ELIGIBLE_ROWS")
    if gate_row["rows_with_eligible_rows_without_value"] > 0:
        failed_checks.append("ELIGIBLE_ROWS_WITHOUT_VALUE")
    if staged_event_sum != silver_row_count:
        failed_checks.append("EVENT_SUM_DIFFERS_FROM_SILVER")

print("Staged rows (repository-days)   :", staged_row_count)
print("Sum of pr_event_count           :", staged_event_sum)
print("Silver rows on the affected days:", silver_row_count)

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| There is no affected day, because Silver has no rows in the scope of this run | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | The affected days are replaced by the staged rows |

How the days are replaced:

- **With a `source_date`:** `INSERT INTO ... REPLACE WHERE activity_date IN (...)` deletes the rows of the affected days and inserts the staged rows. The other days are not touched. Delta also refuses a staged row that belongs to another day.
- **With a blank date:** `INSERT OVERWRITE` replaces the whole table. This also removes a day that no longer exists in Silver.

Both are one Delta commit. A reader sees either the old rows or the new ones, never a half-written day. A repository-day that no longer has events disappears, because its whole day is replaced.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif len(affected_dates) == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    # The first line of the statement depends on the scope of the run.
    if source_date_value is None:
        publish_start = f"INSERT OVERWRITE TABLE {GOLD_TABLE}"
    else:
        publish_start = (
            f"INSERT INTO {GOLD_TABLE} "
            f"REPLACE WHERE activity_date IN ({affected_days_sql})"
        )

    print("Publish statement starts with:", publish_start)

    try:
        spark.sql(f"""
            {publish_start}
            SELECT
                repo_id,
                activity_date,
                repo_name,
                pr_event_count,
                pr_action_known_rows,
                pr_opened_actions,
                pr_closed_actions,
                pr_reopened_actions,
                pr_other_action_rows,
                eligible_closed_rows,
                merged_closed_rows,
                merge_share_pct,
                merged_duration_eligible_rows,
                avg_merge_hours,
                p50_merge_hours,
                closed_duration_eligible_rows,
                avg_close_hours,
                p50_close_hours,
                pr_size_snapshot_rows,
                pr_commits_eligible_rows,
                avg_pr_commits,
                pr_additions_eligible_rows,
                avg_pr_additions,
                pr_deletions_eligible_rows,
                avg_pr_deletions,
                pr_changed_files_eligible_rows,
                avg_pr_changed_files,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These totals describe what this run calculated. They are counted on the staging copy, so they are there also when the run was not published. None of them stops the run.

After a run for all dates they can be compared directly with the expected values in section 9 of the contract.

`pr_size_snapshot_rows` summed over the days counts a pull request once for every day on which it was seen. It is a number of pull-request-days, not of pull requests.

In [0]:
quality_metrics = {
    "repository_days": 0,
    "pr_event_count": 0,
    "pr_action_known_rows": 0,
    "pr_opened_actions": 0,
    "pr_closed_actions": 0,
    "pr_reopened_actions": 0,
    "pr_other_action_rows": 0,
    "eligible_closed_rows": 0,
    "merged_closed_rows": 0,
    "merged_duration_eligible_rows": 0,
    "closed_duration_eligible_rows": 0,
    "pr_size_snapshot_rows": 0,
    "pr_commits_eligible_rows": 0,
    "pr_additions_eligible_rows": 0,
    "pr_deletions_eligible_rows": 0,
    "pr_changed_files_eligible_rows": 0,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            COUNT(*)                                          AS repository_days,
            COALESCE(SUM(pr_event_count), 0)                  AS pr_event_count,
            COALESCE(SUM(pr_action_known_rows), 0)            AS pr_action_known_rows,
            COALESCE(SUM(pr_opened_actions), 0)               AS pr_opened_actions,
            COALESCE(SUM(pr_closed_actions), 0)               AS pr_closed_actions,
            COALESCE(SUM(pr_reopened_actions), 0)             AS pr_reopened_actions,
            COALESCE(SUM(pr_other_action_rows), 0)            AS pr_other_action_rows,
            COALESCE(SUM(eligible_closed_rows), 0)            AS eligible_closed_rows,
            COALESCE(SUM(merged_closed_rows), 0)              AS merged_closed_rows,
            COALESCE(SUM(merged_duration_eligible_rows), 0)   AS merged_duration_eligible_rows,
            COALESCE(SUM(closed_duration_eligible_rows), 0)   AS closed_duration_eligible_rows,
            COALESCE(SUM(pr_size_snapshot_rows), 0)           AS pr_size_snapshot_rows,
            COALESCE(SUM(pr_commits_eligible_rows), 0)        AS pr_commits_eligible_rows,
            COALESCE(SUM(pr_additions_eligible_rows), 0)      AS pr_additions_eligible_rows,
            COALESCE(SUM(pr_deletions_eligible_rows), 0)      AS pr_deletions_eligible_rows,
            COALESCE(SUM(pr_changed_files_eligible_rows), 0)  AS pr_changed_files_eligible_rows
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["repository_days"] = total_row["repository_days"]
    quality_metrics["pr_event_count"] = total_row["pr_event_count"]
    quality_metrics["pr_action_known_rows"] = total_row["pr_action_known_rows"]
    quality_metrics["pr_opened_actions"] = total_row["pr_opened_actions"]
    quality_metrics["pr_closed_actions"] = total_row["pr_closed_actions"]
    quality_metrics["pr_reopened_actions"] = total_row["pr_reopened_actions"]
    quality_metrics["pr_other_action_rows"] = total_row["pr_other_action_rows"]
    quality_metrics["eligible_closed_rows"] = total_row["eligible_closed_rows"]
    quality_metrics["merged_closed_rows"] = total_row["merged_closed_rows"]
    quality_metrics["merged_duration_eligible_rows"] = total_row["merged_duration_eligible_rows"]
    quality_metrics["closed_duration_eligible_rows"] = total_row["closed_duration_eligible_rows"]
    quality_metrics["pr_size_snapshot_rows"] = total_row["pr_size_snapshot_rows"]
    quality_metrics["pr_commits_eligible_rows"] = total_row["pr_commits_eligible_rows"]
    quality_metrics["pr_additions_eligible_rows"] = total_row["pr_additions_eligible_rows"]
    quality_metrics["pr_deletions_eligible_rows"] = total_row["pr_deletions_eligible_rows"]
    quality_metrics["pr_changed_files_eligible_rows"] = total_row["pr_changed_files_eligible_rows"]

print("Repository-days                   :", quality_metrics["repository_days"])
print("Pull request events               :", quality_metrics["pr_event_count"])
print("Events with a known action        :", quality_metrics["pr_action_known_rows"])
print("Opened actions                    :", quality_metrics["pr_opened_actions"])
print("Closed actions                    :", quality_metrics["pr_closed_actions"])
print("Reopened actions                  :", quality_metrics["pr_reopened_actions"])
print("Other actions                     :", quality_metrics["pr_other_action_rows"])
print("Closed events with a merged flag  :", quality_metrics["eligible_closed_rows"])
print("Merged close events               :", quality_metrics["merged_closed_rows"])
print("Merge duration eligible events    :", quality_metrics["merged_duration_eligible_rows"])
print("Close duration eligible events    :", quality_metrics["closed_duration_eligible_rows"])
print("Snapshots (pull-request-days)     :", quality_metrics["pr_size_snapshot_rows"])
print("Snapshots with a commit count     :", quality_metrics["pr_commits_eligible_rows"])
print("Snapshots with added lines        :", quality_metrics["pr_additions_eligible_rows"])
print("Snapshots with deleted lines      :", quality_metrics["pr_deletions_eligible_rows"])
print("Snapshots with changed files      :", quality_metrics["pr_changed_files_eligible_rows"])

if quality_metrics["pr_other_action_rows"] > 0:
    print("WARNING: there are actions other than opened, closed and reopened.")

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    GOLD_TABLE,                                         # gold_table
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    {"pull_request_events": silver_row_count},          # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    published_rows,                                     # gold_rows_published
    gold_table_version,                                 # gold_table_version
    checks_passed,                                      # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. The Gold table was not changed.")

if rows_without_event_date > 0:
    print("WARNING: rows without an event date that were left out:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("repository_delivery_metrics FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 The Gold table per day

Expected on the real week, from the profiling run of 14 November 2026:

| Day | Rows | Pull request events |
|---|---|---|
| 2024-02-01 | 116,405 | 387,295 |
| 2024-02-02 | 113,664 | 332,523 |
| 2024-02-03 | 68,242 | 197,802 |
| 2024-02-04 | 62,025 | 189,714 |
| 2024-02-05 | 122,282 | 388,940 |
| 2024-02-06 | 111,296 | 345,716 |
| 2024-02-07 | 106,572 | 331,968 |

In [0]:
display(spark.sql(f"""
    SELECT
        activity_date,
        COUNT(*)                            AS repository_days,
        SUM(pr_event_count)                 AS pr_events,
        SUM(pr_opened_actions)              AS opened_actions,
        SUM(pr_closed_actions)              AS closed_actions,
        SUM(pr_reopened_actions)            AS reopened_actions,
        SUM(pr_other_action_rows)           AS other_actions,
        SUM(eligible_closed_rows)           AS eligible_closed_rows,
        SUM(merged_closed_rows)             AS merged_closed_rows,
        SUM(merged_duration_eligible_rows)  AS merged_duration_eligible_rows,
        SUM(closed_duration_eligible_rows)  AS closed_duration_eligible_rows,
        SUM(pr_size_snapshot_rows)          AS snapshots
    FROM {GOLD_TABLE}
    GROUP BY activity_date
    ORDER BY activity_date
"""))

### 11.3 The whole Gold table in one row

Expected after a run for all dates on the real week:

| Value | Expected |
|---|---|
| Rows | 700,486 |
| Pull request events | 2,173,958 |
| Opened, closed, reopened, other | 1,126,806 · 1,031,728 · 15,424 · 0 |
| Closed events with a merged flag | 1,031,728 |
| Merged close events | 799,855, which is 77.53% |
| Merge duration eligible rows | 799,855 |
| Close duration eligible rows | 1,031,580, which is 148 lower than the closed actions |
| Snapshots | Not measured before. At least 1,393,696, the number of different pull requests, and at most 2,173,958, the number of events |

The merge share of the whole week is calculated from the two sums. It is not the average of the shares of the rows. The `CASE` keeps the query from dividing by zero when no row has a closed event.

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                            AS repository_days,
        COUNT(DISTINCT activity_date)       AS days,
        SUM(pr_event_count)                 AS pr_events,
        SUM(pr_opened_actions)              AS opened_actions,
        SUM(pr_closed_actions)              AS closed_actions,
        SUM(pr_reopened_actions)            AS reopened_actions,
        SUM(pr_other_action_rows)           AS other_actions,
        SUM(eligible_closed_rows)           AS eligible_closed_rows,
        SUM(merged_closed_rows)             AS merged_closed_rows,
        CASE
            WHEN SUM(eligible_closed_rows) > 0
            THEN ROUND(100 * SUM(merged_closed_rows) / SUM(eligible_closed_rows), 2)
        END                                 AS merge_share_pct_of_all_rows,
        SUM(merged_duration_eligible_rows)  AS merged_duration_eligible_rows,
        SUM(closed_duration_eligible_rows)  AS closed_duration_eligible_rows,
        SUM(pr_size_snapshot_rows)          AS snapshots
    FROM {GOLD_TABLE}
"""))

### 11.4 A sample: the ten repository-days with the most pull request events

A look at real rows. In a row with only one or two events, a share or an average says little. That is why the count columns stand next to them.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY pr_event_count DESC, repo_id, activity_date
    LIMIT 10
"""))

## What this notebook wrote

- `gold.repository_delivery_metrics`: the affected days were replaced, when the status is `SUCCEEDED`.
- `gold_staging.repository_delivery_metrics`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- "Of the closed pull request events seen that day, this share was merged." It is not the repository's lifetime acceptance rate.
- Durations describe pull requests closed in this week. Their start can lie long before the week.
- Look at the median before the average. A few very old pull requests pull the average up.
- 70% of the rows are built from one or two events. A share on them is correct but says little. The count columns allow a reader to set a threshold.